In [1]:
!pip install autogluon.tabular pandas numpy -q
!pip install "git+https://github.com/FutureComputing4AI/EMBER2024.git" -q
!pip install --force-reinstall --no-deps "git+https://github.com/wbond/oscrypto.git" -q

In [4]:
import os
import re
import importlib.util

# ------------------------------------------------------------------
# 1. Patch do oscrypto (detecção de versão do libcrypto)
# ------------------------------------------------------------------
spec = importlib.util.find_spec("oscrypto")
if spec is None:
    raise RuntimeError("oscrypto não está instalado.")

pkg_dir = list(spec.submodule_search_locations)[0]
cffi_path = os.path.join(pkg_dir, "_openssl", "_libcrypto_cffi.py")

with open(cffi_path, "r") as f:
    src = f.read()

if "LibraryNotFoundError('Error detecting the version of libcrypto')" in src:
    src = re.sub(
        r"^([ \t]*)raise LibraryNotFoundError\('Error detecting the version of libcrypto'\)",
        r"\1version_match = re.search(r'(\\d\\.\\d\\.\\d)', '3.0.0')",
        src,
        flags=re.MULTILINE,
    )
    with open(cffi_path, "w") as f:
        f.write(src)
    print("✓ Patch aplicado no oscrypto")
else:
    print("✓ oscrypto não precisa de patch")

# ------------------------------------------------------------------
# 2. Patch do thrember: localizar SignedPEFile no signify instalado
# ------------------------------------------------------------------
sig_spec = importlib.util.find_spec("signify")
thr_spec = importlib.util.find_spec("thrember")
if sig_spec is None or thr_spec is None:
    raise RuntimeError("signify ou thrember não estão instalados.")

sig_dir = list(sig_spec.submodule_search_locations)[0]
sig_parent = os.path.dirname(sig_dir)

novo_modulo = None
for raiz, _, arquivos in os.walk(sig_dir):
    for nome in arquivos:
        if not nome.endswith(".py") or nome == "__init__.py":
            continue
        caminho = os.path.join(raiz, nome)
        with open(caminho, "r", errors="ignore") as f:
            if re.search(r"^class SignedPEFile\b", f.read(), flags=re.MULTILINE):
                rel = os.path.relpath(caminho, sig_parent)
                novo_modulo = rel[:-3].replace(os.sep, ".")
                break
    if novo_modulo:
        break

if novo_modulo is None:
    raise RuntimeError(
        "A classe SignedPEFile não existe nesta versão do signify. "
        "Instale uma versão compatível."
    )

print(f"SignedPEFile encontrado em: {novo_modulo}")

feat_path = os.path.join(list(thr_spec.submodule_search_locations)[0], "features.py")
with open(feat_path, "r") as f:
    feat_src = f.read()

linha_antiga = "from signify.authenticode import SignedPEFile"
linha_nova = f"from {novo_modulo} import SignedPEFile"

if linha_antiga in feat_src:
    feat_src = feat_src.replace(linha_antiga, linha_nova)
    with open(feat_path, "w") as f:
        f.write(feat_src)
    print("✓ Import corrigido no thrember")
else:
    print("✓ thrember não precisa de patch (ou já foi corrigido)")

# ------------------------------------------------------------------
# 3. Imports
# ------------------------------------------------------------------
import numpy as np
import pandas as pd
import thrember
from autogluon.tabular import TabularPredictor, TabularDataset

print("✓ Imports efetuados com sucesso!")

# ------------------------------------------------------------------
# 4. Configurações (Caminho absoluto corrigido)
# ------------------------------------------------------------------
destino = os.path.abspath("./ember2024_data")
os.makedirs(destino, exist_ok=True)

TIPOS_ARQUIVO = ["PDF", "ELF"]
N_TREINO = 200_000      # use None para usar o treino completo
TIME_LIMIT = 1800       # segundos
SEED = 0

# ------------------------------------------------------------------
# 5. Download e vetorização
# ------------------------------------------------------------------
print("Baixando dados do EMBER2024...")
for tipo in TIPOS_ARQUIVO:
    thrember.download_dataset(destino, file_type=tipo)

print("Vetorizando features...")
thrember.create_vectorized_features(destino)

# ------------------------------------------------------------------
# 6. Carregar matrizes vetorizadas
# ------------------------------------------------------------------
X_train, y_train = thrember.read_vectorized_features(destino, subset="train")
X_test, y_test = thrember.read_vectorized_features(destino, subset="test")

# ------------------------------------------------------------------
# 7. Montar DataFrames (float32 para economizar RAM)
# ------------------------------------------------------------------
feature_cols = [f"feat_{i}" for i in range(X_train.shape[1])]

df_train = pd.DataFrame(X_train.astype(np.float32), columns=feature_cols)
df_train["label"] = y_train

df_test = pd.DataFrame(X_test.astype(np.float32), columns=feature_cols)
df_test["label"] = y_test

del X_train, X_test, y_train, y_test

df_train = df_train[df_train["label"].isin([0, 1])].reset_index(drop=True)
df_test = df_test[df_test["label"].isin([0, 1])].reset_index(drop=True)

if N_TREINO is not None and len(df_train) > N_TREINO:
    df_train = df_train.sample(n=N_TREINO, random_state=SEED).reset_index(drop=True)

print(f"Treino: {df_train.shape} | Teste: {df_test.shape}")
print(df_train["label"].value_counts(normalize=True))

train_data = TabularDataset(df_train)
test_data = TabularDataset(df_test)

# ------------------------------------------------------------------
# 8. Treinamento com AutoGluon (Ajustado para caminho local)
# ------------------------------------------------------------------
predictor = TabularPredictor(
    label="label",
    eval_metric="roc_auc",
    problem_type="binary",
    path="./ag_models",
).fit(
    train_data=train_data,
    presets="medium_quality",
    time_limit=TIME_LIMIT,
)

# ------------------------------------------------------------------
# 9. Resultados
# ------------------------------------------------------------------
print("\n=== Resultado da Avaliação ===")
print(predictor.evaluate(test_data))

print("\n=== Leaderboard dos Modelos ===")
print(predictor.leaderboard(test_data))

✓ oscrypto não precisa de patch
SignedPEFile encontrado em: signify.authenticode.signed_file.pe
✓ thrember não precisa de patch (ou já foi corrigido)
✓ Imports efetuados com sucesso!
Baixando dados do EMBER2024...
Unzipping...
Unzipping...
Unzipping...
Unzipping...
Vetorizando features...
Preparing to vectorize raw features
Vectorizing training set


100%|██████████| 156000/156000 [02:31<00:00, 1031.41it/s]


Vectorizing test set


100%|██████████| 36000/36000 [00:44<00:00, 812.15it/s] 


Vectorizing challenge set


100%|██████████| 6315/6315 [00:25<00:00, 250.98it/s] 
Verbosity: 2 (Standard Logging)


... (3 outputs left out)

	1.0	 = Validation score   (roc_auc)
	1354.53s	 = Training   runtime
	1.32s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 404.31s of the 404.30s of remaining time.
	Fitting with cpus=10, gpus=0, mem=2.4/6.0 GB


[1000]	valid_set's binary_logloss: 0.00495096


	Ran out of time, early stopping on iteration 1312. Best iteration is:
	[1312]	valid_set's binary_logloss: 0.00182245
	1.0	 = Validation score   (roc_auc)
	404.65s	 = Training   runtime
	0.17s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the -0.92s of remaining time.
	Fitting 1 model on all data | Fitting with cpus=16, gpus=0, mem=0.0/6.1 GB
	Ensemble Weights: {'LightGBMXT': 1.0}
	1.0	 = Validation score   (roc_auc)
	0.26s	 = Training   runtime
	0.06s	 = Validation runtime
AutoGluon training complete, total runtime = 1818.3s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 1807.1 rows/s (2496 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("d:\Iniciação Científica\ember2024_data\ember2024_data\ag_models")



=== Resultado da Avaliação ===
{'roc_auc': np.float64(0.989332012345679), 'accuracy': 0.9593333333333334, 'balanced_accuracy': np.float64(0.9593333333333334), 'mcc': 0.919504000895407, 'f1': 0.9584468664850136, 'precision': 0.9798050139275766, 'recall': 0.938}

=== Leaderboard dos Modelos ===
                 model  score_test  score_val eval_metric  pred_time_test  \
0             LightGBM    0.990267        1.0     roc_auc        2.098302   
1           LightGBMXT    0.989332        1.0     roc_auc       31.806880   
2  WeightedEnsemble_L2    0.989332        1.0     roc_auc       31.811563   

   pred_time_val     fit_time  pred_time_test_marginal  \
0       0.171240   404.654152                 2.098302   
1       1.319620  1354.529535                31.806880   
2       1.381249  1354.790451                 0.004683   

   pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  \
0                0.171240         404.654152            1       True   
1                1.

In [5]:
import pandas as pd

# 1. Extrai a Importância das Features e o Leaderboard dos Modelos
fi_df = predictor.feature_importance(
    data=test_data[predictor.features() + [predictor.label]],
    model=predictor.model_best,
    subsample_size=1000,
    num_shuffle_sets=1
).reset_index().rename(columns={'index': 'Feature'})

lb_df = predictor.leaderboard(test_data)

# 2. Exporta para um único arquivo Excel com abas separadas
output_filename = "Relatorio_Treinamento_AutoGluon.xlsx"

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    # Aba 1: Importância das Features
    fi_df.to_excel(writer, sheet_name='Feature Importance', index=False)

    # Aba 2: Leaderboard dos Modelos
    lb_df.to_excel(writer, sheet_name='Leaderboard Modelos', index=False)

print(f"✅ Relatório exportado com sucesso para: {output_filename}")

Computing feature importance via permutation shuffling for 692 features using 1000 rows with 1 shuffle sets...
	828.53s	= Expected runtime (828.53s per shuffle set)
	209.11s	= Actual runtime (Completed 1 of 1 shuffle sets)


✅ Relatório exportado com sucesso para: Relatorio_Treinamento_AutoGluon.xlsx
